# Shen et al. (2023) TATN — Tables II, IV, V, VI, VII

Paper-faithful Panasonic NCR18650PF temperature-transfer reproduction. The output compares only **Source pretrained** and **TATN**. Target test labels are evaluated once after training and never select a checkpoint.

Full mode is expensive: 45 Table-II leave-one-out models, 5 transfer source models, and 360 transfer folds. Every completed model and fold is cached on Drive, so rerunning resumes instead of starting over.

In [ ]:
from pathlib import Path
import os, subprocess, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

REPO = Path('/content/battery-soc-transfer-benchmark')
REPO_URL = 'https://' + 'github.com/zhangcastle/battery-soc-transfer-benchmark.git'
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
elif REPO.exists():
    raise RuntimeError(f'Remove incomplete checkout and rerun: {REPO}')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Protocol

- Five temperatures: 25, 10, 0, -10, -20 °C; nine drive cycles each.
- Every tenth raw sample; non-overlapping windows of 90.
- Per-temperature min-max fitted to all nine cycles, exactly as stated in the paper.
- Table II: nine-fold leave-one-cycle-out per temperature; report the eight-cycle training error averaged across folds.
- Transfer source models: train on all nine source-temperature cycles.
- CNN(256×2) + four released-repo BiLSTM layers, hidden size 64, dropout 0.2.
- Adam, lr 1e-3, batch 64, 2000 source and transfer epochs (released-repo defaults).
- Semi-supervised: one test cycle, one non-test cycle with its first 50% labels, seven pseudo-labelled cycles.
- Unsupervised: one test cycle and eight pseudo-labelled cycles.

In [ ]:
import torch
from battery_benchmark.utils.config import load_yaml
from battery_benchmark.reproductions.shen_2023.data import PROFILES, TEMPERATURES, load_temperature_cycles
from battery_benchmark.reproductions.shen_2023.suite import PAPER_TABLES, ShenTableSuite

PATHS = load_yaml(REPO / 'configs/paths/colab_drive.yaml')
RAW_ROOT = Path(PATHS['raw_data_root']) / 'Panasonic_18650PF'
OUTPUT_DIR = Path(PATHS['output_root']) / 'reproductions/shen_2023/tatn_paper_protocol'
FULL_RUN = True  # False: one Cycle_4 fold and 2 epochs, only for plumbing checks
EPOCHS = 2000 if FULL_RUN else 2
TRANSFER_EPOCHS = 2000 if FULL_RUN else 2
RUN_PROFILES = PROFILES if FULL_RUN else ('Cycle_4',)
assert RAW_ROOT.exists(), f'Missing Panasonic data: {RAW_ROOT}'
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('output:', OUTPUT_DIR)
print('profiles:', RUN_PROFILES)

In [ ]:
for temperature in TEMPERATURES:
    cycles = load_temperature_cycles(RAW_ROOT, temperature)
    print(f'{temperature:>3}C:', {name: len(cycle.soc) for name, cycle in cycles.items()})

In [ ]:
suite = ShenTableSuite(
    RAW_ROOT, OUTPUT_DIR, epochs=EPOCHS, transfer_epochs=TRANSFER_EPOCHS,
    batch_size=64, learning_rate=1e-3, lambdas=(1.0, 1.0, 1.0),
    seed=100, device='auto',
)
table_ii = suite.run_table_ii()
display(table_ii.round(3).style.set_caption('Table II — source pretraining errors (%)'))
print('Paper Table II:', PAPER_TABLES['II'])

In [ ]:
# Resumable: each completed fold is written to OUTPUT_DIR/cache/<protocol-id>/folds.
suite.run_all_transfers(profiles=RUN_PROFILES)
print('Transfer folds complete or restored from cache:', len(suite.records))

In [ ]:
tables = suite.tables()
titles = {
    'II': 'Table II — fixed-temperature source pretraining',
    'IV': 'Table IV — 10C source, semi-supervised',
    'V': 'Table V — all sources, semi-supervised',
    'VI': 'Table VI — 10C source, unsupervised',
    'VII': 'Table VII — all sources, unsupervised',
}
for name in ('II', 'IV', 'V', 'VI', 'VII'):
    display(tables[name].round(3).style.set_caption(titles[name] + ' (MAE/RMSE %)'))
    print('Paper TATN reference:', PAPER_TABLES[name])
print('CSV outputs:', sorted(str(path) for path in OUTPUT_DIR.glob('table_*.csv')))

## Reading the result

For Tables IV–VII, compare the `Source pretrained` and `TATN` rows. A lower TATN MAE/RMSE demonstrates effective adaptation. Debug mode does not reproduce paper tables because it uses one fold and two epochs.